# DeepLOB Multi-Target (Mid + Spread + Liquidity)

Notebook to train a DeepLOB encoder + bi-LSTM with multi-head outputs (mid, spread, top-of-book liquidity) and use the predicted spread/liquidity to shape the execution schedule for the last minute.



In [ ]:
import torch
from torch import nn
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)



device: cuda


In [ ]:
# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()
symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)

FEATURE_COLS = [
    "ask_price_1","ask_price_2","ask_price_3","ask_price_4","ask_price_5",
    "bid_price_1","bid_price_2","bid_price_3","bid_price_4","bid_price_5",
    "ask_vol_1","ask_vol_2","ask_vol_3","ask_vol_4","ask_vol_5",
    "bid_vol_1","bid_vol_2","bid_vol_3","bid_vol_4","bid_vol_5",
]

target_window = 60
input_window = 5*60  # last 5 minutes



volume_to_fill: 13736.0


In [ ]:
# DeepLOB encoder and multi-head model
class DeepLOBEncoder(nn.Module):
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=(1,2), stride=(1,1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2),stride=(1,1)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.inp1 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(3,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(5,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3,1),stride=(1,1),padding=(1,0)),
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
    def forward(self, x):
        B, T, F = x.shape
        x = x.view(B*T, 1, 5, 4)
        x = self.conv1(x); x = self.conv2(x); x = self.conv3(x)
        x = torch.cat([self.inp1(x), self.inp2(x), self.inp3(x)], dim=1)
        x = x.permute(0,2,1,3).reshape(x.size(0), x.size(2), -1)
        x = x.mean(dim=1)
        return x.view(B, T, -1)

TARGET_NAMES = ["mid", "spread", "liq"]

class DeepLOBForecastMulti(nn.Module):
    def __init__(self, embed_dim=192, hidden=128, horizon=60, heads=len(TARGET_NAMES)):
        super().__init__()
        self.enc = DeepLOBEncoder()
        self.lstm = nn.LSTM(embed_dim, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden*2, horizon * heads)
        self.horizon = horizon
        self.heads = heads
    def forward(self, x):
        feats = self.enc(x)
        h0 = torch.zeros(2, x.size(0), self.lstm.hidden_size, device=x.device)
        c0 = torch.zeros(2, x.size(0), self.lstm.hidden_size, device=x.device)
        out, _ = self.lstm(feats, (h0,c0))
        last = out[:, -1, :]
        return self.head(last)

def forecast_loss_multi(pred, target, smooth_lambda=0.02, head_weights=(1.0,0.2,0.2)):
    # pred/target: [B, H*3]
    B = pred.size(0)
    pred = pred.view(B, target_window, len(TARGET_NAMES))
    target = target.view(B, target_window, len(TARGET_NAMES))
    weights = torch.tensor(head_weights, device=pred.device, dtype=pred.dtype)
    mse = (pred - target)**2 * weights
    loss = mse.mean()
    # smoothness on mid (head 0)
    mid = pred[:,:,0]
    smooth = (mid[:,1:] - mid[:,:-1])**2
    loss = loss + smooth_lambda * smooth.mean()
    return loss



In [ ]:
# Multi-head dataset
class LastMinuteDatasetMulti(Dataset):
    def __init__(self, x_df: pd.DataFrame, y_df: pd.DataFrame, feat_means, feat_stds, scalers):
        self.samples = []
        for hid, xh in x_df.groupby("anonymized_id"):
            yh = y_df[y_df["anonymized_id"] == hid]
            if len(yh) != target_window:
                continue
            xh = xh.sort_values("time_in_hour").tail(input_window)
            x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
            if x_arr.shape[0] < input_window:
                pad = np.zeros((input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
                x_arr = np.vstack([pad, x_arr])
            mid = ((yh["ask_price_1"] + yh["bid_price_1"]) / 2.0).to_numpy(dtype=np.float32)
            spread = (yh["ask_price_1"] - yh["bid_price_1"]).to_numpy(dtype=np.float32)
            liq = (yh["ask_vol_1"] + yh["bid_vol_1"]).to_numpy(dtype=np.float32)
            if mid.shape[0] != target_window:
                continue
            x_arr = (x_arr - feat_means) / feat_stds
            mid_norm = (mid - scalers['mid_mean']) / scalers['mid_std']
            spread_norm = (spread - scalers['spread_mean']) / scalers['spread_std']
            liq_norm = (liq - scalers['liq_mean']) / scalers['liq_std']
            tgt = np.stack([mid_norm, spread_norm, liq_norm], axis=1)  # [60,3]
            if not np.isfinite(x_arr).all() or not np.isfinite(tgt).all():
                continue
            self.samples.append((x_arr, tgt.reshape(target_window*len(TARGET_NAMES))))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.from_numpy(y)



In [ ]:
# Split + train/val loops (multi-head)
from dataclasses import dataclass

def chrono_split(x_df, y_df, val_ratio=0.2):
    ids = np.sort(x_df["anonymized_id"].unique())
    split = int(len(ids)*(1-val_ratio))
    tr_ids, va_ids = ids[:split], ids[split:]
    x_tr = x_df[x_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    x_va = x_df[x_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_tr = y_df[y_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    y_va = y_df[y_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    return x_tr, x_va, y_tr, y_va

@dataclass
class TrainCfgMulti:
    epochs: int = 15
    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 1e-5
    smooth_lambda: float = 0.02
    val_ratio: float = 0.2
    head_weights = (1.0, 0.2, 0.2)  # mid, spread, liq


def train_val_multi():
    X = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    Y = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)

    # this is where imputation should happen

    x_tr, x_va, y_tr, y_va = chrono_split(X, Y, val_ratio=TrainCfgMulti.val_ratio)

    feat_means = x_tr[FEATURE_COLS].mean().to_numpy().astype(np.float32)
    feat_stds  = x_tr[FEATURE_COLS].std().replace(0,1e-6).to_numpy().astype(np.float32)
    mid_tr = (y_tr["ask_price_1"] + y_tr["bid_price_1"]) / 2.0
    spread_tr = (y_tr["ask_price_1"] - y_tr["bid_price_1"])
    liq_tr = (y_tr["ask_vol_1"] + y_tr["bid_vol_1"])
    scalers = {
        "mid_mean": float(mid_tr.mean()),
        "mid_std": float(mid_tr.std() if mid_tr.std()!=0 else 1e-6),
        "spread_mean": float(spread_tr.mean()),
        "spread_std": float(spread_tr.std() if spread_tr.std()!=0 else 1e-6),
        "liq_mean": float(liq_tr.mean()),
        "liq_std": float(liq_tr.std() if liq_tr.std()!=0 else 1e-6),
    }

    train_ds = LastMinuteDatasetMulti(x_tr, y_tr, feat_means, feat_stds, scalers)
    val_ds   = LastMinuteDatasetMulti(x_va, y_va, feat_means, feat_stds, scalers)
    train_loader = DataLoader(train_ds, batch_size=TrainCfgMulti.batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=TrainCfgMulti.batch_size, shuffle=False)

    model = DeepLOBForecastMulti().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=TrainCfgMulti.lr, weight_decay=TrainCfgMulti.weight_decay)

    for epoch in range(TrainCfgMulti.epochs):
        tr = train_epoch_multi(model, train_loader, opt, TrainCfgMulti)
        va = eval_epoch_multi(model, val_loader, TrainCfgMulti)
        print(f"epoch {epoch+1}: train={tr:.4f} val={va:.4f}")

    scalers.update({"feat_means": feat_means, "feat_stds": feat_stds})
    return model, scalers, val_loader


def train_epoch_multi(model, loader, opt, cfg: TrainCfgMulti):
    model.train(); total=0; n=0
    for xb, yb in loader:
        xb = xb.to(device); yb = yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = forecast_loss_multi(pred, yb, cfg.smooth_lambda, cfg.head_weights)
        loss.backward(); opt.step()
        total += float(loss.item())*xb.size(0); n += xb.size(0)
    return total/max(n,1)


def eval_epoch_multi(model, loader, cfg: TrainCfgMulti):
    model.eval(); total=0; n=0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            pred = model(xb)
            loss = forecast_loss_multi(pred, yb, cfg.smooth_lambda, cfg.head_weights)
            total += float(loss.item())*xb.size(0); n += xb.size(0)
    return total/max(n,1)



In [ ]:
# Train and save scalers/model
run_train_multi = True
if run_train_multi and X_path.exists() and Y_path.exists():
    model_m, scalers_m, val_loader_m = train_val_multi()
else:
    print("Set run_train_multi=True and ensure X_path/Y_path exist.")



epoch 1: train=0.5379 val=0.2517
epoch 2: train=0.5060 val=0.2499
epoch 3: train=0.4706 val=0.2476
epoch 4: train=0.4261 val=0.2453
epoch 5: train=0.3750 val=0.2431
epoch 6: train=0.3190 val=0.2411
epoch 7: train=0.2612 val=0.2394
epoch 8: train=0.2036 val=0.2379
epoch 9: train=0.1517 val=0.2376
epoch 10: train=0.1088 val=0.2393
epoch 11: train=0.0776 val=0.2430
epoch 12: train=0.0572 val=0.2476
epoch 13: train=0.0473 val=0.2522
epoch 14: train=0.0400 val=0.2570
epoch 15: train=0.0328 val=0.2638


In [ ]:
# Head diagnostics: percent error for mid and liquidity (spread ignored)
run_head_check = False  # set True after training to inspect errors
if run_head_check and X_path.exists() and Y_path.exists() and 'model_m' in globals():
    model_m.eval()
    mpe_mid = []; mpe_liq = []; mae_mid = []; mae_liq = []
    eps = 1e-6
    with torch.no_grad():
        for xb, yb in val_loader_m:
            xb = xb.to(device); yb = yb.to(device)
            pred = model_m(xb)  # [B, H*3]
            B = pred.size(0)
            pred = pred.view(B, target_window, len(TARGET_NAMES))
            true = yb.view(B, target_window, len(TARGET_NAMES))
            mid_p = pred[:,:,0].cpu().numpy() * scalers_m['mid_std'] + scalers_m['mid_mean']
            mid_t = true[:,:,0].cpu().numpy() * scalers_m['mid_std'] + scalers_m['mid_mean']
            liq_p = pred[:,:,2].cpu().numpy() * scalers_m['liq_std'] + scalers_m['liq_mean']
            liq_t = true[:,:,2].cpu().numpy() * scalers_m['liq_std'] + scalers_m['liq_mean']
            mae_mid.append(np.abs(mid_p - mid_t).mean())
            mae_liq.append(np.abs(liq_p - liq_t).mean())
            mpe_mid.append((np.abs(mid_p - mid_t) / (np.abs(mid_t)+eps)).mean()*100)
            mpe_liq.append((np.abs(liq_p - liq_t) / (np.abs(liq_t)+eps)).mean()*100)
    if mpe_mid:
        print(f"Val MAE | mid={np.mean(mae_mid):.6f}, liq={np.mean(mae_liq):.6f}")
        print(f"Val %Err | mid={np.mean(mpe_mid):.4f}%, liq={np.mean(mpe_liq):.4f}%")
else:
    if run_head_check:
        print("Train first or set run_head_check=False.")



In [ ]:
# Eval using predicted mid/spread/liquidity to shape the schedule
from data.simulate_walk_the_book import simulate_walk_the_book
# try:
#     from predictive_scheduler import schedule_from_forecasts
# except ImportError:
#     from predictive_scheduler import build_schedule_from_forecasts as schedule_from_forecasts

run_eval_multi = True  # set True to build positions & eval after training
if run_eval_multi and X_path.exists() and Y_path.exists() and 'model_m' in globals():
    X = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    Y = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    ids = np.sort(X["anonymized_id"].unique()); split = int(len(ids)*0.8); va_ids = ids[split:]
    x_va_raw = X[X["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_va_raw = Y[Y["anonymized_id"].isin(va_ids)].reset_index(drop=True)

    errs = []
    for hid in va_ids:
        xh_raw = x_va_raw[x_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        yh_raw = y_va_raw[y_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        if len(yh_raw)!=target_window:
            continue
        feat_means = scalers_m['feat_means']; feat_stds = scalers_m['feat_stds']
        xh = xh_raw.tail(input_window)
        x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
        if x_arr.shape[0] < input_window:
            pad = np.zeros((input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
            x_arr = np.vstack([pad, x_arr])
        x_arr = (x_arr - feat_means) / feat_stds
        xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model_m(xb).cpu().numpy().reshape(target_window, len(TARGET_NAMES))
        # denorm
        mid = pred[:,0] * scalers_m['mid_std'] + scalers_m['mid_mean']
        spread = pred[:,1] * scalers_m['spread_std'] + scalers_m['spread_mean']
        liq = pred[:,2] * scalers_m['liq_std'] + scalers_m['liq_mean']

        # scoring: ignore spread (unstable); prefer high liquidity, price near close forecast
        close_forecast = mid[-1]
        price_score = 1.0 / (np.abs(mid - close_forecast) + 1e-6)
        spread_score = 1.0
        liq_score = np.clip(liq / (np.mean(liq) + 1e-6), 0.0, 5.0)
        score = price_score * liq_score
        score = score / score.sum() if score.sum()!=0 else np.ones_like(score)/len(score)
        sched_forecast = score * volume_to_fill

        def end_window_twap(vol, window=14):
            w = np.zeros(60, dtype=np.float32); w[-window:] = vol / window; return w
        sched_twap = end_window_twap(volume_to_fill, window=14)
        alpha = 0.8  # forecast weight
        positions = alpha * sched_forecast + (1-alpha) * sched_twap
        positions = positions * (volume_to_fill / positions.sum()) if positions.sum()!=0 else sched_twap

        ask_prices = np.column_stack([yh_raw[f"ask_price_{lvl}"].to_numpy(dtype=float) for lvl in range(1,6)])
        ask_vols   = np.column_stack([yh_raw[f"ask_vol_{lvl}"].to_numpy(dtype=float)   for lvl in range(1,6)])
        bid_prices = np.column_stack([yh_raw[f"bid_price_{lvl}"].to_numpy(dtype=float) for lvl in range(1,6)])
        bid_vols   = np.column_stack([yh_raw[f"bid_vol_{lvl}"].to_numpy(dtype=float)   for lvl in range(1,6)])
        total_vol, avg_price = simulate_walk_the_book(positions, ask_prices, ask_vols, bid_prices, bid_vols)
        close_price = yh_raw["close"].dropna().iloc[-1] if len(yh_raw["close"].dropna())>0 else np.nan
        if total_vol>0 and not np.isnan(avg_price) and not np.isnan(close_price):
            rel_err = abs(avg_price - close_price) / close_price
            fill_penalty = min(100.0, volume_to_fill / total_vol)
            err_bps = rel_err * fill_penalty * 10000.0
            errs.append(err_bps)
    if errs:
        print(f"Validation implementation error (bps) over {len(errs)} hours: mean={np.mean(errs):.4f}, median={np.median(errs):.4f}")
    else:
        print("No valid errors computed.")
else:
    print("Set run_eval_multi=True after training to evaluate.")



Validation implementation error (bps) over 2 hours: mean=4.2797, median=4.2797


In [ ]:
# Eval-only (no retrain): set alpha/window and run positions
alpha = 0.8          # forecast weight
end_window = 14      # TWAP window seconds
run_positions_only = True  # set True to evaluate with current model_m/scalers_m

if run_positions_only:
    if 'model_m' not in globals() or 'scalers_m' not in globals():
        print("Train or load model_m + scalers_m first.")
    elif not (X_path.exists() and Y_path.exists()):
        print("Missing X_path/Y_path.")
    else:
        X = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
        Y = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
        ids = np.sort(X["anonymized_id"].unique()); split = int(len(ids)*0.8); va_ids = ids[split:]
        x_va_raw = X[X["anonymized_id"].isin(va_ids)].reset_index(drop=True)
        y_va_raw = Y[Y["anonymized_id"].isin(va_ids)].reset_index(drop=True)

        errs = []
        for hid in va_ids:
            xh_raw = x_va_raw[x_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
            yh_raw = y_va_raw[y_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
            if len(yh_raw)!=target_window:
                continue
            feat_means = scalers_m['feat_means']; feat_stds = scalers_m['feat_stds']
            xh = xh_raw.tail(input_window)
            x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
            if x_arr.shape[0] < input_window:
                pad = np.zeros((input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
                x_arr = np.vstack([pad, x_arr])
            x_arr = (x_arr - feat_means) / feat_stds
            xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = model_m(xb).cpu().numpy().reshape(target_window, len(TARGET_NAMES))
            mid = pred[:,0] * scalers_m['mid_std'] + scalers_m['mid_mean']
            liq = pred[:,2] * scalers_m['liq_std'] + scalers_m['liq_mean']

            close_forecast = mid[-1]
            price_score = 1.0 / (np.abs(mid - close_forecast) + 1e-6)
            liq_score = np.clip(liq / (np.mean(liq) + 1e-6), 0.0, 5.0)
            score = price_score * liq_score
            score = score / score.sum() if score.sum()!=0 else np.ones_like(score)/len(score)
            sched_forecast = score * volume_to_fill

            def end_window_twap(vol, window=end_window):
                w = np.zeros(60, dtype=np.float32); w[-window:] = vol / window; return w
            sched_twap = end_window_twap(volume_to_fill, window=end_window)
            positions = alpha * sched_forecast + (1-alpha) * sched_twap
            positions = positions * (volume_to_fill / positions.sum()) if positions.sum()!=0 else sched_twap

            ask_prices = np.column_stack([yh_raw[f"ask_price_{lvl}"].to_numpy(dtype=float) for lvl in range(1,6)])
            ask_vols   = np.column_stack([yh_raw[f"ask_vol_{lvl}"].to_numpy(dtype=float)   for lvl in range(1,6)])
            bid_prices = np.column_stack([yh_raw[f"bid_price_{lvl}"].to_numpy(dtype=float) for lvl in range(1,6)])
            bid_vols   = np.column_stack([yh_raw[f"bid_vol_{lvl}"].to_numpy(dtype=float)   for lvl in range(1,6)])
            total_vol, avg_price = simulate_walk_the_book(positions, ask_prices, ask_vols, bid_prices, bid_vols)
            close_price = yh_raw["close"].dropna().iloc[-1] if len(yh_raw["close"].dropna())>0 else np.nan
            if total_vol>0 and not np.isnan(avg_price) and not np.isnan(close_price):
                rel_err = abs(avg_price - close_price) / close_price
                fill_penalty = min(100.0, volume_to_fill / total_vol)
                err_bps = rel_err * fill_penalty * 10000.0
                errs.append(err_bps)
        if errs:
            print(f"Validation implementation error (bps) over {len(errs)} hours: mean={np.mean(errs):.4f}, median={np.median(errs):.4f}")
        else:
            print("No valid errors computed.")



Validation implementation error (bps) over 2 hours: mean=4.2797, median=4.2797
